In [11]:
import os
import re
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [21]:
from langchain_community.document_loaders import UnstructuredEPubLoader

In [2]:
load_dotenv()

True

In [1]:
BOOKS_PATH = "data/books"
BOOKS_METADATA = {}

In [12]:
for file in os.listdir(BOOKS_PATH):
    if not os.path.isfile(os.path.join(BOOKS_PATH , file)):
        continue

    name, ext = os.path.splitext(file)

    if " - " in name:
        author, title = name.split(" - ", 1)
    else:
        author, title = "unknown", name

    # parse domain from parentheses e.g. "Man and His Symbols(psychology)"
    domain_match = re.search(r'\((.+?)\)', title)
    if domain_match:
        domain = domain_match.group(1).strip().lower()
        title = re.sub(r'\(.+?\)', '', title).strip()
    else:
        domain = "unknown"

    BOOKS_METADATA[file] = {
        "author": author.strip().lower().replace(" ", "_"),
        "domain": domain,
        "book": title.strip().lower().replace(" ", "_")
    }

In [13]:
BOOKS_METADATA

{'Carl Jung - Man and His Symbols.epub': {'author': 'carl_jung',
  'domain': 'unknown',
  'book': 'man_and_his_symbols'},
 'Carl Jung - Memories, Dreams, Reflections.pdf': {'author': 'carl_jung',
  'domain': 'unknown',
  'book': 'memories,_dreams,_reflections'},
 'Carl Jung - Modern Man in Search of a Soul.epub': {'author': 'carl_jung',
  'domain': 'unknown',
  'book': 'modern_man_in_search_of_a_soul'},
 'Carl Jung - The Collected Works of C. G. Jung, Vol. 9, Part 1 The Archetypes and the Collective Unconscious.epub': {'author': 'carl_jung',
  'domain': 'unknown',
  'book': 'the_collected_works_of_c._g._jung,_vol._9,_part_1_the_archetypes_and_the_collective_unconscious'},
 'Carl Jung - The Red Book.pdf': {'author': 'carl_jung',
  'domain': 'unknown',
  'book': 'the_red_book'},
 'Carl Jung - Man and His Symbols(psychology).epub': {'author': 'carl_jung',
  'domain': 'psychology',
  'book': 'man_and_his_symbols'},
 'Carl Jung - Memories, Dreams, Reflections(psychology).pdf': {'author': 'c

In [19]:
list(BOOKS_METADATA.keys())[0]

'Carl Jung - Man and His Symbols.epub'

In [22]:
file_path = os.path.join(BOOKS_PATH, list(BOOKS_METADATA.keys())[0])
loader = UnstructuredEPubLoader(file_path)

In [6]:
books_list = [file for file in os.listdir("data/books")]
books_list

['Carl Jung - Man and His Symbols.epub',
 'Carl Jung - Memories, Dreams, Reflections.pdf',
 'Carl Jung - Modern Man in Search of a Soul.epub',
 'Carl Jung - The Collected Works of C. G. Jung, Vol. 9, Part 1 The Archetypes and the Collective Unconscious.epub',
 'Carl Jung - The Red Book.pdf']

In [7]:

CHROMA_PATH = "chroma_db"

BOOK_METADATA = {
    books_list[1]: {
        "author": "jung",
        "domain": "psychology",
        "book": "Memories, Dreams, Reflections"
    },
}

In [8]:
BOOK_METADATA

{'Carl Jung - Memories, Dreams, Reflections.pdf': {'author': 'jung',
  'domain': 'psychology',
  'book': 'Memories, Dreams, Reflections'}}

In [23]:
def load_and_chunk(pdf_path: str, metadata: dict) -> list:
    """
    Loads a PDF file and splits it into chunks of text.
    """
    print(f"Loading: data/books/{pdf_path}")
    loader = PyPDFLoader(f"data/books/{pdf_path}")
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", " "]        # hierachical splitting
    )
    chunks = splitter.split_documents(documents)

    for chunk in chunks:
        chunk.metadata.update(metadata)

    print(f"  Created {len(chunks)} chunks")
    return chunks

In [24]:
chunks1 = load_and_chunk(books_list[1], BOOK_METADATA[books_list[1]])
chunks1[:10]

Loading: data/books/Carl Jung - Memories, Dreams, Reflections.pdf
  Created 1109 chunks


[Document(metadata={'producer': 'Recoded by LuraDocument PDF v2.16', 'creator': 'PyPDF', 'creationdate': '2006-10-29T19:18:54+00:00', 'moddate': '2022-03-26T03:46:14-05:00', 'source': 'data/books/Carl Jung - Memories, Dreams, Reflections.pdf', 'total_pages': 449, 'page': 0, 'page_label': '1', 'author': 'jung', 'domain': 'psychology', 'book': 'Memories, Dreams, Reflections'}, page_content='MEMORIES,\nDREAMS,\nREFLECTIONS\nby C. G.Jung\nRECORDED AND EDITED BY\nAniela Jaffe\nTRANSLATED FROM THE GERMAN BY\nRichard and Clara Winston\nREVISED EDITION\nVINTAGE BOOKS\nADivisionofRandomHouse, Inc.\nNEW YORK'),
 Document(metadata={'producer': 'Recoded by LuraDocument PDF v2.16', 'creator': 'PyPDF', 'creationdate': '2006-10-29T19:18:54+00:00', 'moddate': '2022-03-26T03:46:14-05:00', 'source': 'data/books/Carl Jung - Memories, Dreams, Reflections.pdf', 'total_pages': 449, 'page': 1, 'page_label': '2', 'author': 'jung', 'domain': 'psychology', 'book': 'Memories, Dreams, Reflections'}, page_content=

In [13]:
len(chunks1)

1256

In [38]:
i =0

for chunk in chunks1:
    i += 1
    if i <= 3:
        print(chunk.page_content)
        print(chunk.metadata.get("page"))
    

MEMORIES,
DREAMS,
REFLECTIONS
by C. G.Jung
RECORDED AND EDITED BY
Aniela Jaffe
TRANSLATED FROM THE GERMAN BY
Richard and Clara Winston
REVISED EDITION
VINTAGE BOOKS
ADivisionofRandomHouse, Inc.
NEW YORK
0
VINTAGE BOOKS EDITION, APRIL1989
Copyright 1961, 1962, 1963byRandom House, Inc.
All rights reserved under International and Pan-American Copyright
Conventions. Published in the United States byRandom House, Inc.,
New York. Distributed in Canada by Random House of Canada
Limited, Toronto. Originally publishedby Pantheon Books, a division
ofRandom House, Inc., in 1963. Final revised edition in hardcover
publishedbyPantheon Books, February1973.
Originallypublishedunder the title "Erinnerungen
Traume Gedanken."
LibraryofCongress Cataloging-in-Publication Data
Jung, C. G. (Carl Gustav), 1875-1961.
[Erinnerungen, Traume, Gedanken. English]
Memories, dreams, reflections/by C.G. Jung;recordedandedited
byAnielaJaffe;translatedfrom theGermanbyRichardand Clara
Winston. Rev. ed.
p. cm.
Originally

In [47]:
for chunk in chunks1:
    if chunk.metadata.get("page") == 6:
        print(chunk.page_content)
        print('divider')
        print(len(chunk.page_content))

Introduction
me that all the 'outer* aspects ofmy life should be accidental.
Only what is interior has proved to have substance and a
determining value. As a result, all memory of outer events has
faded, and perhaps these 'outer' experiences were never so very
essential anyhow, or were so only in that they coincided with
phases of my inner development. An enormous part of these
'outer* manifestations ofmy life has vanished from my memory
for the very reason, so it has seemed to me, that I partici-
pated in them with all my energies. Yet these are the very
things that make up a sensible biography: persons one has met,
travels, adventures, entanglements, blows of destiny, and so on.
But with few exceptions all these things have become for me
phantasms which I barely recollect and which my mind has no
desire to reconstruct, for they no longer stirmy imagination.
"On the other hand, my recollection of 'inner' experiences
has grown all the more vivid and colorful. This poses a problem
of de

In [48]:
comb = [chunk.page_content for chunk in chunks1 if chunk.metadata.get("page") == 4] + [chunk.page_content for chunk in chunks1 if chunk.metadata.get("page") == 5]

In [49]:
comb

['Introduction\nmade to the book. In January 1959 he was at his country house\nin Bollingen. He devoted every morning to reading chosen\nchapters of our book, which had meanwhile been hammered\ninto shape. When he returned the chapter, "On Life after\nDeath/\'he said to me, "Something withinme has been touched.\nA gradient has formed, and I must write/\' Such was the origin\nof "Late Thoughts," inwhich he voiced his deepest and perhaps\nhis most far-reaching convictions.\nIn thesummer of thatsame year of 1959, likewise in Bollingen,\nJung wrote the chapter on Kenya and Uganda. The section on\nthe Pueblo Indians is taken from an unpublished and un-\nfinished manuscript that deals with general questions of the\npsychology of primitives.\nIn order to complete the chapters "Sigmund Freud" and\n"Confrontation with the Unconscious/\' I incorporated a number\nof passages from a seminar delivered in 1925, in which Jung\nspoke for the first time of his inner development,\nThe chapter "Psychiatr

In [ ]:
def ingest_book(pdf_filename: str):
    if pdf_filename not in BOOK_METADATA:
        print(f"No metadata found for {pdf_filename}. Add it to BOOK_METADATA first.")
        return

    pdf_path = os.path.join(BOOKS_PATH, pdf_filename)
    if not os.path.exists(pdf_path):
        print(f"File not found: {pdf_path}")
        return

    metadata = BOOK_METADATA[pdf_filename]
    chunks = load_and_chunk(pdf_path, metadata)

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    vectorstore = Chroma(
        collection_name="author_council",
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH
    )

In [50]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(
    collection_name="author_council",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)

print(vectorstore._collection.count())

1256
